In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch14_웹 데이터 수집</font>

# 1. BeautifulSoup / parser
    (정적 웹크롤링, 공공 api 사용)

`pip install bs4` (아나콘다 설치 시 자동으로 설치되는 패키지에 포함됨)


▪ 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/ <br>
▪ Documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [3]:
import requests # HTTP 요청을 처리하는 라이브러리

from requests_file import FileAdapter

In [5]:
# 로컬 파일을 웹요청 하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑처로 처리
response = s.get("file:///ai/source/01_python/data/ch14_sample.html") # c:ai/source/01_python/data/ch14_sample.html
response

<Response [200]>

In [6]:
if response:
    print('해당 url에 접근함')
else:
    print('url 접근 거부됨')

해당 url에 접근함


In [7]:
response.status_code
# 200 : 정상, 404 : 없는 페이지

200

In [9]:
response.content # html의 디코딩되지 않은 순수 바이너리(bytes) 데이터

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [10]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [11]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [14]:
# HTML 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# soup

In [15]:
# 1_soup.select_one('선택자') : 해당 선택자 중 첫 엘리먼트
el = soup.select_one('h1.greeting.css')

In [24]:
print('el : ', el)
print('el.text : ', el.text)
print('el.string : ', el.string)
print('el의 속성들 :', el.attrs)
print('el의 클래스 속성 :', el.attrs['class'])
print('el의 클래스 속성 :', el.attrs.get('class'))
print('el의 href 속성 :', el.attrs.get('href')) # get() : 데이터가 없을 경우 None 반환
print('el의 이름 :', el.name)

el :  <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text :  Hello, CSS
el.string :  Hello, CSS
el의 속성들 : {'class': ['greeting', 'css'], 'id': 'text'}
el의 클래스 속성 : ['greeting', 'css']
el의 클래스 속성 : ['greeting', 'css']
el의 href 속성 : None
el의 이름 : h1


In [31]:
# 2_soup.select('선택자') : 해당 선택자 전체 엘리먼트 / 없을 경우 빈 list 반환
els = soup.select('h1.css')

In [36]:
print('el : ', els)
print('els.text : ',[el.text for el in els])
print('els.string : ',[el.string for el in els])
print('els의 속성들 : ',[el.attrs for el in els])
print('els의 클래스 속성들 : ',[el.attrs for el in els])
print('els의 이름들 : ',[el.name for el in els])

el :  [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els.text :  ['Hello, CSS', 'Hi, CSS']
els.string :  ['Hello, CSS', 'Hi, CSS']
els의 속성들 :  [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els의 클래스 속성들 :  [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els의 이름들 :  ['h1', 'h1']


In [42]:
# 3_soup.find(태그, 속성) : 조건에 맞는 첫 번째 엘리먼트
print('soup.find :', soup.find('h1'),{'class':'css'})
print('soup.find :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('soup.find :', soup.find('h1',{'id':'text'}))

soup.find : <h1 class="greeting css" id="text">Hello, CSS</h1> {'class': 'css'}
soup.find : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
soup.find : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [51]:
#4 soup.find_all(태그, 속성) : 조건에 맞는 전체 엘리먼트 / 없을 경우 빈 list 반환
print('모든 h1.css 태그와 span 태그 :', soup.select('h1.css.greeting, span'))
print('모든 h1.css 태그와 span 태그 (span 태그 호출 실패) :', \
      soup.find_all(['h1', 'span'], class_=['css'])) # 태그 && 속성 ; span 출력 x
print()
print('모든 h1.css 태그와 span 태그 :', soup.find_all(['h1', 'span'], class_='css')
                                    + soup.find_all('span'))

모든 h1.css 태그와 span 태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css 태그와 span 태그 (span 태그 호출 실패) : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]

모든 h1.css 태그와 span 태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [52]:
# 존재하지 않는 엘리먼트를 찾을 때의 반환값 규칙
print('find_all(empty list) :', soup.find_all('a')) # empty list : false return
print('find(None) :', soup.find('a')) # None return
print('select(empty list) :', soup.select('a')) # empty list : false return
print('select_one(None) :', soup.select_one('a')) # None return

find_all(empty list) : []
find(None) : None
select(empty list) : []
select_one(None) : None


# 2. 정적 웹 데이터 수집
- 정적 웹 크롤링

## 2.1 html 웹 데이터 수집
- beautifulSoup 모듈 활용

### 1) 환율 정보 가져오기
- 네이버 증권 > 시장지표 : https://finance.naver.com/marketindex/
* 크롤링 허용 범위는 사이트의 ~/robots.txt 에서 확인 가능
    - Allow : 크롤링 허용 가능
    - Disallow : 크롤링 제한 폴더

In [56]:
# soup 객체 생성 방법 1. requests
import requests
from bs4 import BeautifulSoup

url = 'https://finance.naver.com/marketindex/'

response = requests.get(url) # Response 객체

soup = BeautifulSoup(response.text, 'html.parser')

In [63]:
# soup 객체 생성 방법 2. urlopen
from urllib.request import urlopen

url = 'https://finance.naver.com/marketindex/'

response = urlopen(url)

soup = BeautifulSoup(response, 'html.parser')

In [ ]:
# 종류 : h3.h_lst > span.blind
# 환율 : div.head_info > span.value
# 변동률 : div.head_info > span.change
# 증감: div.head_info > span.blind

In [91]:
#div.head_info 밑의 span.value (find 계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

[1417.7, 889.17, 1635.6, 210.09, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.92, 4441.1, 200855.05]


In [95]:
# span.value (find 계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

[1417.7, 889.2, 1635.6, 210.1, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 200855.0]


In [104]:
el1 = soup.select_one('h3.h_lst > span.blind')
el2 = float(soup.select_one('div.head_info > span.value').text.replace(',',''))
el3 = soup.select_one('div.head_info > span.change')
el4 = soup.select_one('div.head_info > span.blind')

In [105]:
print(f'{el1.text} 환율 :{el2}원 / {el3.text}% {el4.text}')

미국 USD 환율 :1417.7원 / 4.20% 상승


- 변수 생성 및 데이터 읽어오기
- 데이터 다듬고 사용 용도에 따라 타입 변환하기

In [168]:
title_els = soup.select('h3.h_lst > span.blind')
unit_els = soup.select('div.head_info > span > span.blind')
trand_els = soup.select('div.head_info > span.blind')

In [169]:
# 현재가, 변동폭
values = [round(float(el.text.replace(',','')), 1) for el in soup.select('div.head_info > span.value')]
changes = [round(float(el.text.replace(',','')), 3) for el in soup.select('div.head_info > span.change')]

In [170]:
# 단위들 : 'div.head_info > span > span.blind'
len(unit_els) # >>> 11
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '') # 단위가 없는 데이터에 '' 추가
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [171]:
# 지표명, 증감
titles = [title_el.string for title_el in title_els]
trands = [trand_el.text for trand_el in trand_els]

- 데이터 프레임으로 전환하기

In [178]:
import pandas as pd

In [180]:
# 첫 번째 방식 (딕셔너리 리스트)
data = []
for title, value, unit, change, trand in zip(titles, values, units, changes, trands):
    # print(f'{title} : {value}{unit} / {change} - {trand}')
    data.append({
        '지표명' : title,
        '현재가' : value,
        '단위' : unit,
        '변동폭' : change,
        '증감' : trand
    })

pd.DataFrame(data)

,지표명,현재가,단위,변동폭,증감
0,미국 USD,1417.7,원,4.200,상승
1,일본 JPY(100엔),889.2,원,2.210,상승
2,유럽연합 EUR,1635.6,원,4.490,상승
3,중국 CNY,210.1,원,0.580,상승
4,달러/일본 엔,159.2,엔,0.280,상승
5,유로/달러,1.2,달러,0.001,하락
6,영국 파운드/달러,1.4,달러,0.001,하락
7,달러인덱스,99.7,,0.010,상승
8,WTI,83.2,달러,1.070,상승
9,휘발유,1863.9,원,0.210,하락


In [181]:
# 두 번째 방식 (2차원 리스트)
data = []
for title, value, unit, change, trand in zip(titles, values, units, changes, trands):
    data.append([title, value, unit, change, trand])
pd.DataFrame(data, columns=['title', 'value', 'unit', 'change', 'trand'])

,title,value,unit,change,trand
0,미국 USD,1417.7,원,4.200,상승
1,일본 JPY(100엔),889.2,원,2.210,상승
2,유럽연합 EUR,1635.6,원,4.490,상승
3,중국 CNY,210.1,원,0.580,상승
4,달러/일본 엔,159.2,엔,0.280,상승
5,유로/달러,1.2,달러,0.001,하락
6,영국 파운드/달러,1.4,달러,0.001,하락
7,달러인덱스,99.7,,0.010,상승
8,WTI,83.2,달러,1.070,상승
9,휘발유,1863.9,원,0.210,하락


### 2) 로또 번호 가져오기
- User Agent 추가하여 soup 생성하기
- https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto (daum의 lotto 검색 결과)

In [4]:
# soup 객체 생성 방법 1. requests
import requests
from bs4 import BeautifulSoup

url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'

response = requests.get(url) # Response 객체

soup = BeautifulSoup(response.text, 'html.parser')

response.status_code

200

In [5]:
# soup 객체 생성 방법 2. urlopen
from urllib.request import urlopen, Request

url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'
# F12 - Network - Headers - User-Agent
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# import Request 후 url과 headers을 변수에 저장
request = Request(url, headers=headers)
# 저장한 변수로 urlopen
response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

response.status #다음(Daum) 서버가 파이썬 기본 요청(Python-urllib)을 봇으로 감지하고 차단하여 soup 객체 정상 생성 실패

200

- 데이터 읽어오기

In [6]:
episode_el = soup.select_one('span.f_red')

episode = episode_el.string

episode

'1236회'

In [7]:
title_el = soup.select_one('div.prize > strong')

title = title_el.text[-4:]

title

'당첨번호'

In [8]:
lottonum_els = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')

lottonums = [lottonum.string for lottonum in lottonum_els]

lottonums

['12', '18', '21', '29', '34', '38']

In [9]:
bonus_el = soup.select_one('span.ball > span.screen_out')

bonus = bonus_el.string

bonus

'보너스'

In [10]:
bonusnum_el = soup.select_one('div.lottonum > span.bg_ball1')

bonusnum = bonusnum_el.string

bonusnum

'10'

In [11]:
time_el = soup.select_one('div.prize > span.date')

time = time_el.string

time

'(2026.08.08 추첨)'

In [12]:
print(episode, time)
print(title, [int(lottonum) for lottonum in lottonums])
print(bonus, [int(bonusnum)])

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 [10]


- find 계열로 변경해보기

In [266]:
episode_el = soup.find('span', class_ ='f_red')

episode = episode_el.string

episode

'1236회'

In [270]:
prize = soup.find('div', class_='prize')
title_el = prize.find('strong')

title = title_el.text[-4:]

title

'당첨번호'

In [274]:
divlotto = soup.find('div', class_='lottonum')
lottonum_els = divlotto.find_all('span', class_='ball')[:-2]

lottonums = [lottonum.string for lottonum in lottonum_els]

lottonums

['12', '18', '21', '29', '34', '38']

In [279]:
txt_bonus = soup.find('span', class_='txt_bonus')
bonus_el = txt_bonus.find('span', class_='screen_out')

bonus = bonus_el.string

bonus

'보너스'

In [280]:
bonusnum_el = divlotto.find('span', class_='bg_ball1')

bonusnum = bonusnum_el.string

bonusnum

'10'

In [281]:
time_el = prize.find('span', class_='date')

time = time_el.string

time

'(2026.08.08 추첨)'

In [284]:
print(episode, time)
print(title, [int(lottonum) for lottonum in lottonums])
print(bonus, [int(bonusnum)])

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 [10]


### 3) 다음 뉴스 검색 리스트
```
no title    href
0  타이틀1  http://~
1  타이틀2  http://~
2  타이틀3  http://~
```

In [22]:
# 방법 1
import requests
from bs4 import BeautifulSoup

word = '경주기행'
url = f'https://search.daum.net/search?w=news&q={word}'

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

In [23]:
# 방법 2
from urllib.request import urlopen, Request
from urllib.parse import quote

word = quote('비트코인')
url = f'https://search.daum.net/search?w=news&q={word}'

# headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)

request = Request(url)
request.add_header('User-Agent','Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')

response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

In [24]:
# 뉴스 링크 : div.item-title > strong.tit-g.clamp-g > a

items_find_list = []

items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')

for idx, item in enumerate(items_el):
    items_find_list.append({'no' : idx+1, 'title' : item.text, 'link' : item.attrs.get('href')})
    
import pandas as pd
pd.DataFrame(items_find_list)

""


In [27]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 list return'
    
    url = f'https://search.daum.net/search?w=news'
    params = {'q':keyword, 'p':page}
    
    headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
    response = requests.get(url, headers=headers, params=params)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    items_find_list = []
    items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')

    for idx, item in enumerate(items_el):
        items_find_list.append({'no' : idx+1+(page-1)*10, 'title' : item.text, 'link' : item.attrs.get('href')})
    
    return items_find_list

In [29]:
collect_list('오디세이', 5)

import pandas as pd
pd.DataFrame(items_find_list)

""
